In [19]:
import os
import re
import glob
import pandas as pd
from functools import reduce

def merge_diff_datasets(DIR):
    # ============================================================
    # Directory containing CSV result files
    # ============================================================
    #DIR = "m_2"   # <-- change this

    # ============================================================
    # Regex patterns
    # ============================================================

    # Extract lambda from filename
    lambda_pattern = re.compile(r"lambdda-([0-9.]+)\.csv")

    # Extract class (m, n, sigma) from instance filename
    # Example:
    # mglcs_10_100_4_1.txt.out
    # -> (10, 100, 4)
    instance_pattern = re.compile(
        r"mglcs_(\d+)_(\d+)_(\d+)_\d+\.txt\.out"
    )

    # ============================================================
    # Store aggregated dataframes
    # ============================================================
    dfs = []

    # ============================================================
    # Process each CSV file
    # ============================================================
    for filepath in sorted(glob.glob(os.path.join(DIR, "*.csv"))):

        filename = os.path.basename(filepath)
        #print(filename)
        # --------------------------------------------------------
        # Extract lambda value
        # --------------------------------------------------------
        lambda_match = lambda_pattern.search(filename)

        if not lambda_match:
            print(f"Skipping file (lambda not found): {filename}")
            continue

        lambda_value = lambda_match.group(1)

        print(f"Processing lambda = {lambda_value}")

        # --------------------------------------------------------
        # Read CSV
        # --------------------------------------------------------
        df = pd.read_csv(filepath)

        # --------------------------------------------------------
        # Extract (m, n, sigma)
        # --------------------------------------------------------
        extracted = df["file"].str.extract(instance_pattern)

        df["m"] = extracted[0].astype(int)
        df["n"] = extracted[1].astype(int)
        df["sigma"] = extracted[2].astype(int)

        # --------------------------------------------------------
        # Aggregate by graph class
        # --------------------------------------------------------
        grouped = (
            df.groupby(["m", "n", "sigma"])["quality"]
            .mean()
            .reset_index()
        )

        # Rename quality column to lambda value
        grouped = grouped.rename(columns={"quality": lambda_value})

        dfs.append(grouped)
    return dfs

In [20]:
# ============================================================
# Merge all aggregated dataframes
# ============================================================

def merge_dfs(dfs):
    
    merged_df = reduce(
        lambda left, right: pd.merge(
            left,
            right,
            on=["m", "n", "sigma"],
            how="outer"
        ),
        dfs
    )

    # ============================================================
    # Sort rows
    # ============================================================
    merged_df = merged_df.sort_values(
        by=["m", "n", "sigma"]
    ).reset_index(drop=True)

    # ============================================================
    # Save result
    # ============================================================
    #output_file = "aggregated_lambda_results.csv"

    #merged_df.to_csv(output_file, index=False)

    #print("\nFinal merged dataframe:")
    #print(merged_df)
 
    #print(f"\nSaved to: {output_file}")
    return merged_df
    

In [21]:
dfs_m2 = merge_diff_datasets("m_2")
dfs_m3 = merge_diff_datasets("m_3")
dfs_m5 = merge_diff_datasets("m_5")
dfs_m10 = merge_diff_datasets("m_10")

## merged results:
merged_df_m2 = merge_dfs(dfs_m2)
merged_df_m3 = merge_dfs(dfs_m3)
merged_df_m5 = merge_dfs(dfs_m5)
merged_df_m10 = merge_dfs(dfs_m10)

##filter out acc. to m:

merged_df_m2_filter = merged_df_m2[merged_df_m2["m"] == 2]
merged_df_m3_filter = merged_df_m3[merged_df_m3["m"] == 3]
merged_df_m5_filter = merged_df_m5[merged_df_m5["m"] == 5]
merged_df_m10_filter = merged_df_m10[merged_df_m10["m"] == 10]


Processing lambda = 0.00
Processing lambda = 0.05
Processing lambda = 0.1
Processing lambda = 0.15
Processing lambda = 0.2
Processing lambda = 0.25
Processing lambda = 0.3
Processing lambda = 0.35
Processing lambda = 0.4
Processing lambda = 0.45
Processing lambda = 0.5
Processing lambda = 0.55
Processing lambda = 0.60
Processing lambda = 0.65
Processing lambda = 0.70
Processing lambda = 0.75
Processing lambda = 0.80
Processing lambda = 0.85
Processing lambda = 0.90
Processing lambda = 0.95
Processing lambda = 1.0
Processing lambda = 0.00
Processing lambda = 0.05
Processing lambda = 0.10
Processing lambda = 0.15
Processing lambda = 0.20
Processing lambda = 0.25
Processing lambda = 0.30
Processing lambda = 0.35
Processing lambda = 0.40
Processing lambda = 0.45
Processing lambda = 0.50
Processing lambda = 0.55
Processing lambda = 0.60
Processing lambda = 0.65
Processing lambda = 0.70
Processing lambda = 0.75
Processing lambda = 0.80
Processing lambda = 0.85
Processing lambda = 0.90
Proces

In [22]:
merged_df_m2_filter["0.60"]  #lambda = 0.6 (for m=2)


0     37.7
1     30.1
2     72.9
3     62.1
4    140.7
5    125.8
6    299.5
7    310.9
Name: 0.60, dtype: float64

In [23]:
merged_df_m3_filter["0.50"]
#merged_df_m10_filter.mean()

8      31.2
9      22.9
10     58.5
11     48.7
12    106.9
13     98.0
14    157.2
15    238.3
Name: 0.50, dtype: float64

In [24]:
merged_df_m5_filter["0.35"] # lambda = 0.35 (for m=5)


16    20.5
17    15.3
18    32.8
19    27.9
20    46.6
21    42.4
22    45.6
23    56.7
Name: 0.35, dtype: float64

In [25]:
merged_df_m10_filter["0.15"] # lambda = 0.15 (for m=10)

24    11.6
25     7.5
26    14.4
27     9.9
28    14.8
29     9.5
30    14.6
31    10.1
Name: 0.15, dtype: float64

In [60]:
merged_df_m2_filter.mean()

m          2.0000
n        212.5000
sigma      3.0000
0.00     124.6375
0.05     126.3375
0.1      132.3125
0.15     131.8375
0.2      131.8375
0.25     133.4625
0.3      133.4375
0.35     134.1500
0.4      134.5000
0.45     134.6625
0.5      133.9000
0.55     134.1875
0.60     134.9625
0.65     134.4125
0.70     134.2375
0.75     132.4875
0.80     133.1625
0.85     132.0750
0.90     131.8500
0.95     132.0125
1.0      131.2125
dtype: float64

## Benchmark set REAL

In [67]:
#LIMSBS-3000_imbs-iters-5000_num-roots-10_U10_5_5_A2_F2_lambdda-0.00_real.csv


def diff_datasets_real(DIR):
    # ============================================================
    # Directory containing CSV result files
    # ============================================================
    #DIR = "m_2"   # <-- change this

    # ============================================================
    # Regex patterns
    # ============================================================

    # Extract lambda from filename
    lambda_pattern = re.compile(r"lambdda-([0-9.]+)_real\.csv")

    # Extract class (m, n, sigma) from instance filename
    # Example:
    # mglcs_10_100_4_1.txt.out
    # -> (10, 100, 4)

    #4_100_600.rat_lambda_0.00.out
    instance_pattern = re.compile(
        r"(\d+)_(\d+)_(\d+)\.(rat|virus)\_lambda\_(\d+\.\d+)\.out"
    )

    # ============================================================
    # Store aggregated dataframes
    # ============================================================
    dfs = []

    # ============================================================
    # Process each CSV file
    # ============================================================
    for filepath in sorted(glob.glob(os.path.join(DIR, "*.csv"))):

        filename = os.path.basename(filepath)
        #print(filename)
        # --------------------------------------------------------
        # Extract lambda value
        # --------------------------------------------------------
        lambda_match = lambda_pattern.search(filename)

        if not lambda_match:
            print(f"Skipping file (lambda not found): {filename}")
            continue

        lambda_value = lambda_match.group(1)

        print(f"Processing lambda = {lambda_value}")

        # --------------------------------------------------------
        # Read CSV
        # --------------------------------------------------------
        df = pd.read_csv(filepath)

        # --------------------------------------------------------
        # Extract (m, n, sigma)
        # --------------------------------------------------------
        extracted = df["file"].str.extract(instance_pattern)
        
        df["sigma"] = extracted[0].astype(int)
        df["m"] = extracted[1].astype(int)
        df["n"] = extracted[2].astype(int)
        df["benchmark"] = extracted[3] 
        
        df.drop(columns=["file"], axis=1, inplace=True)
        #df.drop(columns=["time"], axis=1, inplace=True)

        df = df.rename(columns={"quality": lambda_value})
        df = df.rename(columns={"time": lambda_value + "_time"})


        dfs.append(df)
    return dfs



def merge_real_dfs(dfs):
    
    merged_df = reduce(
        lambda left, right: pd.merge(
            left,
            right,
            on=["m", "n", "sigma", "benchmark"],
            how="outer"
        ),
        dfs
    )

    # ============================================================
    # Sort rows
    # ============================================================
    merged_df = merged_df.sort_values(
        by=["m", "n", "sigma", "benchmark"]
    ).reset_index(drop=True)

    return merged_df


In [68]:
real_alphas = diff_datasets_real("real_results")

real_merged = merge_real_dfs(real_alphas)


Processing lambda = 0.00
Processing lambda = 0.05
Processing lambda = 0.10
Processing lambda = 0.15
Processing lambda = 0.20
Processing lambda = 0.25
Processing lambda = 0.30
Processing lambda = 0.35
Processing lambda = 0.40
Processing lambda = 0.45
Processing lambda = 0.50
Processing lambda = 0.55
Processing lambda = 0.60
Processing lambda = 0.65
Processing lambda = 0.70
Processing lambda = 0.75
Processing lambda = 0.80
Processing lambda = 0.85
Processing lambda = 0.90
Processing lambda = 0.95
Processing lambda = 1.00


In [69]:
real_merged

,0.00_time,0.00,sigma,m,n,benchmark,0.05_time,0.05,0.10_time,0.10,...,0.80_time,0.80,0.85_time,0.85,0.90_time,0.90,0.95_time,0.95,1.00_time,1.00
0,33.9568,27,4,10,600,rat,37.1410,27,38.7561,26,...,48.4295,25,37.4790,27,38.93510,26,46.8768,26,49.1079,28
1,71.7429,67,4,10,600,virus,80.0582,68,76.9241,84,...,175.2500,83,113.4870,82,89.42100,81,112.5490,79,136.4110,79
2,15.7910,11,4,15,600,rat,23.3737,11,19.3912,11,...,21.4976,11,20.3292,13,21.63170,13,27.8558,13,28.2635,13
3,21.6048,19,4,15,600,virus,26.7080,19,28.3411,19,...,27.8494,18,23.3026,19,17.91000,19,30.6378,19,32.4625,19
4,13.9496,10,4,20,600,rat,16.8730,10,18.0927,10,...,18.1055,10,14.8978,10,9.55153,11,19.8443,11,19.6767,11
5,19.0946,10,4,20,600,virus,23.5808,10,27.1544,10,...,24.3917,10,20.3298,10,26.05890,10,30.4302,10,21.4779,10
6,20.3713,7,4,25,600,rat,23.1259,7,27.2022,7,...,24.7338,9,20.4235,9,26.27550,9,28.9982,9,21.5391,9
7,21.0181,10,4,25,600,virus,24.0925,10,28.2082,10,...,20.2783,10,19.4654,12,27.22580,12,31.1902,12,22.3682,12
8,22.3152,5,4,40,600,rat,25.0468,5,29.4942,5,...,21.7639,5,18.1150,5,28.21100,5,32.5148,5,23.5423,5
9,17.7252,7,4,40,600,virus,19.4688,7,26.4992,7,...,25.6529,7,24.1857,7,28.13150,7,35.1305,7,24.4100,7


In [70]:
real_merged.mean()

/tmp/ipykernel_15501/3748541568.py:1: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  real_merged.mean()


0.00_time     25.859825
0.00           9.850000
sigma          4.000000
m             70.000000
n            600.000000
0.05_time     29.807390
0.05           9.900000
0.10_time     31.321285
0.10          10.650000
0.15_time     32.301525
0.15           9.950000
0.20_time     32.287175
0.20           9.850000
0.25_time     36.772885
0.25          10.950000
0.30_time     32.193875
0.30          10.300000
0.35_time     32.949745
0.35          10.050000
0.40_time     40.522265
0.40          10.550000
0.45_time     31.519055
0.45          10.850000
0.50_time     29.837555
0.50          10.700000
0.55_time     32.358045
0.55          10.300000
0.60_time     35.660535
0.60           9.850000
0.65_time     41.723040
0.65          11.000000
0.70_time     36.597925
0.70          10.150000
0.75_time     37.278905
0.75          10.250000
0.80_time     37.706485
0.80          10.600000
0.85_time     30.304150
0.85          10.900000
0.90_time     31.470796
0.90          10.850000
0.95_time     39

In [72]:
real_merged[["sigma", "m", "n", "benchmark", "0.65", "0.65_time"]]

,sigma,m,n,benchmark,0.65,0.65_time
0,4,10,600,rat,26,69.5797
1,4,10,600,virus,84,163.5900
2,4,15,600,rat,12,40.8151
3,4,15,600,virus,18,25.2138
4,4,20,600,rat,13,16.8977
5,4,20,600,virus,12,24.1305
6,4,25,600,rat,7,24.3080
7,4,25,600,virus,12,25.1708
8,4,40,600,rat,5,19.2536
9,4,40,600,virus,7,23.4114
